# Analysis of generated initial conditions

TODO

In [ ]:
import numpy as np
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1 import make_axes_locatable

import sys
root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stepsic.parameters import CosmoParameters
from stepsic.data import CosmoData
from stepsic.field import cubic_voxels, create_grid

In [ ]:
import logging
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
# Facecolor values from S. Conradi @S_Conradi/@profConradi
custom_settings = {
    'figure.facecolor': '#ffffff',
    # 'figure.facecolor': '#f4f0e8',
    'axes.facecolor': '#ffffff',
    # 'axes.facecolor': '#f4f0e8',
    'axes.edgecolor': '0.3',
    'axes.linewidth' : '0.5',
    'axes.grid': False,
    'grid.color': '0.7',
    'grid.linestyle': ':',
    'grid.alpha': 0.6,
    'xtick.bottom': True,
    'xtick.top': True,
    'ytick.left': True,
    'ytick.right': True,
}
for t in ['xtick', 'ytick']:
    custom_settings[f'{t}.direction'] = 'in'
    custom_settings[f'{t}.color'] = '0.3'
    for m in ['major', 'minor']:
        custom_settings[f'{t}.{m}.width'] = 0.5
        custom_settings[f'{t}.{m}.size'] = 6 if m == 'major' else 3
sns.set_theme(palette=sns.color_palette('deep', as_cmap=False),
              rc=custom_settings)
plt.rcParams['text.usetex'] = False

In [ ]:
def inspect_glass_file(filename):
    with h5py.File(filename, 'r') as f:
        print('File structure:')
        def print_structure(name, obj):
            print(f'  {name}: {type(obj).__name__}')
            if hasattr(obj, 'shape'):
                print(f'    Shape: {obj.shape}')
            if hasattr(obj, 'dtype'):
                print(f'    Dtype: {obj.dtype}')
        f.visititems(print_structure)
        
        def check_attribute(name):
            if 'Header' in f and name in f['Header'].attrs:
                print(f"{name}: {f['Header'].attrs[name]}")

        # Check specific attributes
        check_attribute('BoxSize')
        check_attribute('HubbleParam')
        check_attribute('Omega0')
        check_attribute('OmegaLambda')
        check_attribute('OmegaBaryon')
        check_attribute('NpartTotal')
        check_attribute('MassTable')
        
        if 'PartType1/Coordinates' in f:
            coords = f['PartType1/Coordinates'][:]
            print(f'Coordinates shape: {coords.shape}')
            print(f'First few particles:\n{coords[:3]}')

In [ ]:
def histogram(x, bins=50):
    '''TODO'''
    hist, edge = np.histogram(x, bins=bins)
    bin_c = (edge[:-1] + edge[1:]) / 2
    bin_w = np.diff(edge)  # Width of each bin
    return bin_c, hist, bin_w

In [ ]:
params = CosmoParameters(path=Path('..', 'config.toml')).get_parameters()

In [ ]:
params['REDSHIFT'] = 49
params['SCALE'] = 1 / (1 + params['REDSHIFT'])

nvox, dk = cubic_voxels(params['NMESH'], params['LBOX'])
pos, _ = create_grid(nvox, dk)
ic_orig = CosmoData(pos=pos.astype(params['DTYPE']))
#ic_orig = CosmoData.load_snapshot(Path(params['INPUT_GLASS']))
# ic_orig.to_internal_units(params)
# ic_orig.rescale_snapshot_mass(params)
# ic_orig.periodic_shift(params)

path = Path(f'../output/stepsic_Lx1000_Ly1000_Lz1000_R3D500_D4D75_z{params["REDSHIFT"]}.hdf5')
ic = CosmoData.load_snapshot(path)
#ic.to_internal_units(params)
#ic.rescale_snapshot_mass(params)
#ic.periodic_shift(params)
inspect_glass_file(path)

path = Path(f'../output/ics_gadget_2lpt_z{params["REDSHIFT"]}.hdf5')
ic_test = CosmoData.load_snapshot(path)
ic_test.pos += dk/2 - params['LBOX'] / 2
#ic_test.to_internal_units(params)
inspect_glass_file(path)

In [ ]:
4100 / ic.mass[0]

In [ ]:
6.6376e11 / 1.4495e12

Min particle mass =     1.4495e+12 M_sun
Max particle mass =     1.6702e+13 M_sun
Jelenlegi stepsic2:
Min particle mass =     6.6376e+11 M_sun
Max particle mass =     7.6481e+12 M_sun

In [ ]:
Lbox = params['LBOX']
lpt = params['LPTORDER']
unit = 'Mpc/h' if params['HINDEPENDENT'] else 'Mpc'
unitk = 'kpc/h' if params['HINDEPENDENT'] else 'kpc'

axis_limits = np.array([
    [0, l] if p else [-l/2, l/2]
    for p, l in zip(params['PERIODIC'], params['LBOX'])
])

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)

labels = ['x', 'y', 'z']
for i, ax in enumerate(axes.flat[:3]):
    ax.set_aspect(1)
    idx = [k for k in range(3) if k != i] + [i]

    x, y, z = ic.pos.T[idx]
    mask = np.abs(z - z.mean()) < 20
    ax.scatter(x[mask], y[mask], s=0.1**2, c='k', alpha=0.5)
    ax.set_xlim(axis_limits[idx[0]])
    ax.set_ylim(axis_limits[idx[1]])
    ax.set_xlabel(f'{labels[idx[0]]} [{unit}]', fontsize=10)
    ax.set_ylabel(f'{labels[idx[1]]} [{unit}]', fontsize=10)
plt.show()

In [ ]:
ic.mass - ic_orig.mass

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
dis = np.linalg.norm((ic.pos - ic_orig.pos)*1000, axis=1)
c, h, w = histogram(dis, bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.5)
ax.set_yscale('log')
ax.set_xlabel(f'Displacement [{unitk}]', fontsize=10)
ax.set_title(f'stepsic2.0 {lpt}-LPT displacements [z={params["REDSHIFT"]}]', loc='left', fontsize=10)

ax = next(axes)
vel = np.linalg.norm((ic.vel - ic_orig.vel), axis=1)
c, h, w = histogram(vel, bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.5)
ax.set_xlabel(r'Velocity [km/s]', fontsize=10)
ax.set_title(f'stepsic2.0 {lpt}-LPT velocities [z={params["REDSHIFT"]}]', loc='left', fontsize=10)

ax = next(axes)
c, h, w = histogram((ic.mass - ic_orig.mass), bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:blue', alpha=0.5)
ax.set_xlabel(r'Mass [1e11 Msol]', fontsize=10)
ax.set_title(f'stepsic2.0 {lpt}-LPT masses [z={params["REDSHIFT"]}]', loc='left', fontsize=10)

fpath = Path('output', f'stepsic-{lpt}lpt-z{params["REDSHIFT"]}-analysis-stepsic2.png')
fig.savefig(fpath, bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
dis = np.linalg.norm((ic_test.pos - ic_orig.pos)*1000, axis=1)
c, h, w = histogram(dis, bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.5)
ax.set_yscale('log')
ax.set_xlabel(f'Displacement [{unitk}]', fontsize=10)
ax.set_title(f'monofonic {lpt}-LPT displacements [z={params["REDSHIFT"]}]', loc='left', fontsize=10)

ax = next(axes)
vel = np.linalg.norm((ic_test.vel - ic_orig.vel), axis=1)
c, h, w = histogram(vel, bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.5)
ax.set_xlabel(r'Velocity [km/s]', fontsize=10)
ax.set_title(f'monofonic {lpt}-LPT velocities [z={params["REDSHIFT"]}]', loc='left', fontsize=10)

ax = next(axes)
c, h, w = histogram(ic_test.mass - ic_orig.mass, bins=100)
ax.bar(c, h, w, lw=0.5, color='tab:red', alpha=0.5)
ax.set_xlabel(r'Mass [1e11 Msol]', fontsize=10)
ax.set_title(f'monofonic {lpt}-LPT masses [z={params["REDSHIFT"]}]', loc='left', fontsize=10)

fpath = Path('output', f'stepsic-{lpt}lpt-z{params["REDSHIFT"]}-analysis-monofonic.png')
fig.savefig(fpath, bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
vel1 = np.linalg.norm((ic.pos - ic_orig.pos), axis=1)
c1, h1, _ = histogram(vel1, bins=100)
vel2 = np.linalg.norm((ic_test.pos - ic_orig.pos), axis=1)
c2, h2, _ = histogram(vel2, bins=100)
c2 / c1

In [ ]:
vel1 = np.linalg.norm((ic.vel - ic_orig.vel), axis=1)
c1, h1, _ = histogram(vel1, bins=100)
vel2 = np.linalg.norm((ic_test.vel - ic_orig.vel), axis=1)
c2, h2, _ = histogram(vel2, bins=100)
c2 / c1

#### Calculate $P(k)$ from simulation

In [ ]:
import subprocess

In [ ]:
stepsic_odir = Path('/home/masterdesky/github/stepsic/output/')
files = [
    Path(stepsic_odir, 'monofonic_2lpt_z15.gadget'),
    Path(stepsic_odir, 'monofonic_2lpt_z49.gadget'),
    Path(stepsic_odir, 'monofonic_2lpt_z63.gadget'),
    Path(stepsic_odir, 'stepsic_Lx1000_Ly1000_Lz1000_R3D500_D4D75_z15.gadget'),
    Path(stepsic_odir, 'stepsic_Lx1000_Ly1000_Lz1000_R3D500_D4D75_z49.gadget'),
    Path(stepsic_odir, 'stepsic_Lx1000_Ly1000_Lz1000_R3D500_D4D75_z63.gadget')
]

genpk_exe = Path('/home/masterdesky/github/GenPK/gen-pk')
genpk_odir = Path('/home/masterdesky/github/GenPK/output/')

kpk = {}
for f in files:
    subprocess.run([genpk_exe, '-i', str(f), '-o', str(genpk_odir)])
    kpk[f.name] = {}
    kpk[f.name]['k'], kpk[f.name]['pk'], _ = np.genfromtxt(Path(genpk_odir, f'PK-DM-{f.name}')).T

for kpki in kpk.values():
    kpki['k'] *= 2*np.pi**2 / np.min(params['LBOX'])
    kpki['pk'] *= np.min(params['LBOX'])**3 / (2*np.pi**3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=120)

for fi, kpki in kpk.items():
    k, pk = kpki['k'], kpki['pk']
    ic_gen = fi.split('_')[0]
    lw = 2 if ic_gen == 'stepsic' else 2
    ls = ':' if ic_gen == 'stepsic' else '-'
    ax.loglog(k[k < 2], pk[k < 2], label=fi, ls=ls, lw=lw)
ax.set_xlabel(r'k [h Mpc$^{-1}$]')
ax.set_ylabel(r'P(k) [Mpc$^3$ h$^{-3}$]')
ax.legend(loc=(0.02, 0.02), fontsize=6, frameon=False)
fig.savefig('output/power-spectrum-monofonic-vs-stepsic.png', bbox_inches='tight')
plt.show()

In [ ]:
from nbodykit.lab import ArrayCatalog, ArrayMesh, FieldMesh, FFTPower

In [ ]:
from stepsic.field import \
    white_noise, generate_delta_k, cubic_voxels
from stepsic.cosmology import \
    hubble_a, F_omega, F2_omega, CAMBCosmology, ColossusCosmology

In [ ]:
def pk_particle(
        x, nvox, Lbox, *, resampler='tsc', interlaced=True):
    '''TODO'''
    catalog = ArrayCatalog({'x': x}, BoxSize=Lbox)
    mesh = catalog.to_mesh(
        Nmesh=nvox,
        resampler=resampler,
        interlaced=interlaced,  # Cancel the Alias effect in Fourier modes
        compensated=True,  # Corrects for MAS smoothing
        position='x'
    )
    kmin = 2.0 * np.pi / np.max(Lbox) # Fundamental mode
    kmax = np.sqrt(3) * np.pi * nvox[0] / Lbox[0]  # Nyquist frequency
    r = FFTPower(mesh, mode='1d', dk=kmin, kmin=kmin, kmax=kmax)  # TODO: PR for `kmax`
    return r.power

In [ ]:
def pk_field(field, Lbox):
    '''TODO'''
    nvox = field.shape
    mesh = ArrayMesh(field, BoxSize=Lbox)
    kmin = 2.0 * np.pi / np.max(Lbox) # Fundamental mode
    kmax = np.sqrt(3) * np.pi * nvox[0] / Lbox[0]  # Nyquist frequency
    r = FFTPower(mesh, mode='1d', dk=kmin/2, kmin=kmin, kmax=kmax)  # TODO: PR for `kmax`
    return r.power

In [ ]:
# Initialize cosmology models and calculate growth parameters
cosmo_colossus = ColossusCosmology(
    H0=params['H0'], Om0=params['OMEGA_M'], Ob0=params['OMEGA_B'],
    Ol0=params['OMEGA_L'], sigma8=params['SIGMA8'], ns=params['NS'],
    Neff=params['NNU'], w0=params['W0'], wa=params['WA'], Tcmb0=1e-6)
g1 = 1
D1 = g1 * cosmo_colossus.Dzplus0(params['REDSHIFT'])
g2 = - 3.0/7.0 * params['OMEGA_M']**(-1/143)
D2 = g2 * D1**2  # Bernardeau et al. 2002, eq. 97  # Unused!
log.info(f"D1(z={params['REDSHIFT']}) = {D1:.6f}")
log.info(f"D2(z={params['REDSHIFT']}) = {D2:.6f}")

# Bernardeau et al. 2002, eq. 99
# velocity prefactors (a*H*f) should be in km/s/Mpc
Hz = hubble_a(params['SCALE'], params['H0'], params['OMEGA_M'], params['OMEGA_L'])
log.info(f'Initial Hubble parameter: {Hz} km/s/Mpc')
aHf1 = params['SCALE'] * Hz * F_omega(params['SCALE'], params['OMEGA_M'], params['OMEGA_L'])
aHf2 = params['SCALE'] * Hz * F2_omega(params['SCALE'], params['OMEGA_M'], params['OMEGA_L'])
log.info(f"1st vel. prefac(z={params['REDSHIFT']}) = {aHf1:.6f}")
log.info(f"2nd vel. prefac(z={params['REDSHIFT']}) = {aHf2:.6f}")

# Construct the linear power spectrum and backscale it to `z`
if params['SPECTRUM'] == 'camb':
    cosmo_camb = CAMBCosmology(
        H0=params['H0'], ombh2=params['OMBH2'], omch2=params['OMCH2'],
        omk=params.get('OMK', 0.0), mnu=params['MNU'], nnu=params['NNU'],
        YHe=params['YHE'], TCMB=params['TCMB'], zrei=params['ZREI'],
        w0=params['W0'], wa=params['WA'], nonlinear=False)
    kh, pk, pk3 = cosmo_camb.get_spectrum(
        z=0, As=params['AS'], ns=params['NS'], sigma8_init=params['SIGMA8'],
        kmin=1/np.min(params['LBOX']), kmax=100, npoints=2048)
    pk = pk[0]*D1**2  # Backscale P(k,z=0) with D1^2 to desired `z`
elif params['SPECTRUM'] == 'input':
    # Should contain 2 rows or columns: log(k) and a scaled log(P^3(k))
    kh_log, pk3_log = np.genfromtxt(params['INPUT_SPECTRUM'])
    kh, pk3 = np.exp(kh_log), np.exp(pk3_log)
    pk = pk3 / (kh**3/(2*np.pi**2))

In [ ]:
nmesh = 16 #params['NMESH']

nvox, dk = cubic_voxels(nmesh, params['LBOX'])
# White noise field for complete reproducibility
field = white_noise(nvox=nvox, seed=params['SEED'])
with h5py.File(Path(params['IC_DIR'], 'initial_conditions.hdf5'), 'w') as f:
    f.create_dataset('ic_white_noise', data=np.fft.irfftn(field))
delta_k = generate_delta_k(kh, pk, nvox, dk, field=field)

### 1-LPT

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
ax.set_box_aspect(1)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt = pk_particle(ic.pos, nvox, Lbox)
ax.loglog(pk_pt['k'], pk_pt['power'].real, color='tab:red', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('Matter power spectrum of perturbed positions', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
ax.set_box_aspect(1)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_delta = pk_field(np.fft.irfftn(delta_k), Lbox)
ax.loglog(pk_delta['k'], pk_delta['power'].real, color='tab:green', lw=1.5, label='Field P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title(r'Matter power spectrum of $\delta(x)$', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/pk_grid_N{nmesh}_L{np.max(Lbox)}_1lpt.png', bbox_inches='tight', dpi=300)
plt.show()

### 2-LPT

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
ax.set_box_aspect(1)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt = pk_particle(ic.pos, nvox, Lbox)
ax.loglog(pk_pt['k'], pk_pt['power'].real, color='tab:red', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('Matter power spectrum (2-LPT)', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
ax.set_box_aspect(1)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_delta = pk_field(np.fft.irfftn(delta_k), Lbox)
ax.loglog(pk_delta['k'], pk_delta['power'].real, color='tab:green', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_title(r'Matter power spectrum of $\delta(x)$', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/pk_grid_N{nmesh}_L{np.max(Lbox)}_2lpt.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
nr, nc = 1, 2
fig, axes = plt.subplots(nr, nc, figsize=(nc*6, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt1 = pk_particle(xpert_lpt1, nvox, Lbox)
ax.loglog(pk_pt1['k'], pk_pt1['power'].real,
          color='tab:red', lw=1.5, label='Particle P(k)')
pk_pt1 = pk_particle(xpert_m1, nvox, Lbox)
ax.loglog(pk_pt1['k'], pk_pt1['power'].real,
          color='tab:green', lw=1.5, ls='--', label='MonofonIC P(k)')
ax.set_xlim(None, 2.0)
ax.set_ylim(1e-3, None)
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('MonofonIC vs StePS (1-LPT))', loc='left', fontsize=10)
ax.legend(loc='lower left', fontsize=10, frameon=False)

ax = next(axes)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt2 = pk_particle(xpert_lpt2, nvox, Lbox)
ax.loglog(pk_pt2['k'], pk_pt2['power'].real,
          color='tab:blue', lw=1.5, label='Particle P(k)')
pk_pt2 = pk_particle(xpert_m2, nvox, Lbox)
ax.loglog(pk_pt2['k'], pk_pt2['power'].real,
          color='tab:orange', lw=1.5, ls='--', label='MonofonIC P(k)')
ax.set_xlim(None, 2.0)
ax.set_ylim(1e-3, None)
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('MonofonIC vs StePS (2-LPT)', loc='left', fontsize=10)
ax.legend(loc='lower left', fontsize=10, frameon=False)

fig.savefig(f'output/pk_grid_N{nmesh}_L{np.max(Lbox)}_compare.png', bbox_inches='tight', dpi=300)
plt.show()

### 1-LPT vs 2-LPT

In [ ]:
nr, nc = 1, 3
fig, axes = plt.subplots(nr, nc, figsize=(nc*5, nr*4), dpi=120)
axes = iter(axes.flat)

ax = next(axes)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt1 = pk_particle(xpert_lpt1, nvox, Lbox)
ax.loglog(pk_pt1['k'], pk_pt1['power'].real,
          color='tab:red', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_ylabel(r'$P(k)\,[h^{-3}\mathrm{Mpc}^3]$', fontsize=10)
ax.set_title('Matter power spectrum (1-LPT)', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
ax.loglog(kh, pk, color='k', lw=1.5, label='CAMB P(k)', alpha=0.5)
pk_pt2 = pk_particle(xpert_lpt2, nvox, Lbox)
ax.loglog(pk_pt2['k'], pk_pt2['power'].real,
          color='tab:blue', lw=1.5, label='Particle P(k)')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_title('Matter power spectrum (2-LPT)', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

ax = next(axes)
ax.plot(pk_pt2['k'], pk_pt2['power'].real - pk_pt1['power'].real,
        color='tab:green', lw=1.5, label=r'$P_{\mathrm{2LPT}}(k) - P_{\mathrm{1LPT}}(k)$')
ax.set_xlabel('$k\,[h/\mathrm{Mpc}]$', fontsize=10)
ax.set_title('Difference between 1-LPT and 2-LPT', loc='left', fontsize=10)
ax.legend(loc='upper right', fontsize=10, frameon=False)

plt.savefig(f'output/pk_grid_N{nmesh}_L{np.max(Lbox)}_compare.png', bbox_inches='tight', dpi=300)
plt.show()